# Treinamento Python — Nível 7 — Tratamento de Erros

> **Data Engineering Track** | Python do zero ao Data Engineering
> Cada seção termina com um **exercício de fixação** e ao final há um **desafio integrador**.

---
# 📘 Aula 01 Excecoes

## NÍVEL 7 — Tratamento de Erros | Aula 1: Exceções

## 1. O problema sem tratamento de erros

int("abc")       → ValueError
10 / 0           → ZeroDivisionError
lista[100]       → IndexError
dict["nao_existe"]  → KeyError
None.upper()     → AttributeError

## 2. try / except básico

In [ ]:
try:
    resultado = 10 / 0
except ZeroDivisionError:
    print("Erro: divisão por zero!")

# Capturando múltiplos tipos
def converter_para_int(valor):
    try:
        return int(valor)
    except ValueError:
        print(f"'{valor}' não é um número válido")
        return None
    except TypeError:
        print(f"Tipo inválido: {type(valor)}")
        return None

print(converter_para_int("42"))    # 42
print(converter_para_int("abc"))   # mensagem de erro + None
print(converter_para_int(None))    # mensagem de erro + None

## 3. try / except / else / finally

In [ ]:
def ler_arquivo(caminho):
    try:
        with open(caminho, "r") as f:
            conteudo = f.read()
    except FileNotFoundError:
        print(f"Arquivo não encontrado: {caminho}")
        return None
    except PermissionError:
        print(f"Sem permissão para ler: {caminho}")
        return None
    else:
        # else executa APENAS se não ocorreu exceção
        print(f"Arquivo lido com sucesso: {len(conteudo)} caracteres")
        return conteudo
    finally:
        # finally SEMPRE executa — ideal para limpeza de recursos
        print("Tentativa de leitura concluída.")

resultado = ler_arquivo("/arquivo_inexistente.txt")

## 4. Capturando a exceção como objeto

In [ ]:
def processar_json(texto):
    import json
    try:
        dados = json.loads(texto)
        return dados
    except json.JSONDecodeError as e:
        print(f"JSON inválido na linha {e.lineno}, coluna {e.colno}: {e.msg}")
        return {}
    except Exception as e:
        # Exception captura qualquer exceção — use com cuidado
        print(f"Erro inesperado: {type(e).__name__}: {e}")
        return {}

processar_json('{"nome": "Ana"}')          # OK
processar_json('{nome: "Ana"}')            # JSONDecodeError

## 5. raise — lançando exceções

In [ ]:
def dividir(a, b):
    if b == 0:
        raise ValueError("Divisor não pode ser zero")
    return a / b

try:
    print(dividir(10, 2))
    print(dividir(10, 0))
except ValueError as e:
    print(f"Erro de valor: {e}")

# Re-lançando a exceção após tratar
def processar_pagamento(valor):
    try:
        if valor <= 0:
            raise ValueError("Valor deve ser positivo")
        print(f"Processando pagamento de R${valor:.2f}")
    except ValueError:
        print("Log: tentativa de pagamento inválido")
        raise   # re-lança a mesma exceção para o chamador tratar

try:
    processar_pagamento(-100)
except ValueError as e:
    print(f"Capturado pelo chamador: {e}")

## EXERCÍCIO DE FIXAÇÃO 7.1

Crie uma função robusta de divisão de lista em lotes (batches):
- Recebe uma lista e um tamanho de lote
- Valida que tamanho > 0 (raise ValueError)
- Valida que lista não está vazia (raise ValueError)
- Retorna lista de listas
- Trate todos os casos de erro com mensagens claras

In [ ]:
# Escreva seu código aqui


---
# 📘 Aula 02 Excecoes Customizadas E Logging

## NÍVEL 7 — Tratamento de Erros | Aula 2: Exceções Customizadas e Logging

In [ ]:
import logging
from pathlib import Path

## 1. Exceções customizadas

Crie exceções específicas para seu domínio — facilita debug e tratamento

In [ ]:
class DataPipelineError(Exception):
    """Erro base para todos os erros do pipeline."""
    pass

class ExtractionError(DataPipelineError):
    """Falha na extração de dados."""
    def __init__(self, fonte, mensagem):
        self.fonte = fonte
        super().__init__(f"[{fonte}] {mensagem}")

class ValidationError(DataPipelineError):
    """Dados inválidos encontrados."""
    def __init__(self, campo, valor, regra):
        self.campo = campo
        self.valor = valor
        super().__init__(f"Campo '{campo}' inválido: valor={valor!r} | Regra: {regra}")

class TransformationError(DataPipelineError):
    """Falha na transformação de dados."""
    pass


# Usando exceções customizadas
def extrair_dados(fonte):
    if fonte == "offline":
        raise ExtractionError(fonte, "Fonte indisponível — connection refused")
    return [{"id": 1, "valor": "abc"}, {"id": 2, "valor": "42"}]

def validar_registro(registro):
    try:
        valor = int(registro["valor"])
    except ValueError:
        raise ValidationError("valor", registro["valor"], "deve ser inteiro")
    return {**registro, "valor": valor}

# Pipeline com tratamento por tipo de exceção
try:
    dados = extrair_dados("online")
    validados = []
    for r in dados:
        try:
            validados.append(validar_registro(r))
        except ValidationError as e:
            print(f"  IGNORADO: {e}")

    print(f"Registros válidos: {validados}")
except ExtractionError as e:
    print(f"Falha de extração: {e}")
except DataPipelineError as e:
    print(f"Erro no pipeline: {e}")

## 2. Módulo logging — a forma profissional de registrar logs

In [ ]:
LOG_DIR = Path("/home/lg/Documents/personal/projects/treinamento/python/nivel_7/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Configurando o logger
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler(LOG_DIR / "pipeline.log", encoding="utf-8"),
        logging.StreamHandler()   # também exibe no terminal
    ]
)

logger = logging.getLogger("pipeline.etl")

# Níveis de log (em ordem de severidade)
logger.debug("Iniciando verificação de configurações")          # dev/debug
logger.info("Pipeline iniciado com 1500 registros")             # informação normal
logger.warning("12 registros nulos foram ignorados")            # atenção necessária
logger.error("Falha ao conectar em db.empresa.com")             # erro recuperável
logger.critical("Disco cheio — pipeline abortado")              # erro crítico

## 3. Loggers por módulo — padrão recomendado

In [ ]:
def criar_logger(nome, nivel=logging.INFO):
    log = logging.getLogger(nome)
    log.setLevel(nivel)
    return log

extractor_log = criar_logger("pipeline.extractor")
transform_log = criar_logger("pipeline.transformer")

def extrair(fonte, logger):
    logger.info(f"Conectando à fonte: {fonte}")
    try:
        if fonte == "indisponivel":
            raise ConnectionError("Timeout após 30s")
        logger.info("Extração concluída: 500 registros")
        return list(range(500))
    except ConnectionError as e:
        logger.error(f"Falha de conexão: {e}")
        return []

dados = extrair("postgres://db:5432/vendas", extractor_log)
transform_log.info(f"Recebidos {len(dados)} registros para transformação")

## EXERCÍCIO DE FIXAÇÃO 7.2

Construa um sistema de validação de schema para dados de pipeline:
- Classe SchemaError (customizada)
- Função validar_schema(registro, schema) onde schema define
  tipo esperado para cada campo: {"nome": str, "idade": int, "ativo": bool}
- Logue erros com logging
- Processe uma lista de registros, separando válidos de inválidos

In [ ]:
# Escreva seu código aqui


---
# 🏆 Desafio Nivel 7

## NÍVEL 7 — DESAFIO FINAL | Pipeline Resiliente com Tratamento de Erros Profissional

CONTEXTO:
Você vai construir um pipeline de dados que:
- Falha graciosamente em vez de travar
- Registra todos os eventos em log
- Separa registros válidos de inválidos
- Gera relatório de qualidade dos dados

In [ ]:
# Escreva seu código aqui
